# 05b — PyTorch training and image classification — Practice

Read the lesson and run the example. Implement each TODO, then run its checks. A NotImplementedError is expected until you write your answer. Try for 20–30 minutes before opening a hint; open the solution only after recording an attempt. Copy this notebook into work/ to keep your own version.

**Prerequisites:** earlier lessons in the course order. **Environment:** Install requirements/deep.txt. CPU is sufficient.

[Stage guide](../stages/05.md) · [Course map](../PLAN.md) · [Project](../../projects/stage05/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Use tensors, autograd, modules, and loaders
- Separate train and evaluation modes
- Train a small CPU image classifier


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Before the exercises: Training a PyTorch model one batch at a time

Read this in small pieces. Explain each new term in your own words, trace the example on paper, and change one input before continuing. If the arithmetic is unfamiliar, use [the math bridge](../MATH_START_HERE.md).

### Concepts in plain language

A tensor is an array with shape, type, and device. Autograd records operations so it can calculate derivatives. Parameters are tensors the optimizer updates. A Dataset supplies examples; a DataLoader groups them into batches, optionally shuffling training examples.

An epoch is one pass through the training data. For each batch: clear old gradients, compute logits and loss, backpropagate, and call the optimizer step. Logits are unnormalized class scores; CrossEntropyLoss takes logits and integer class labels, so do not apply softmax first.

An MLP uses dense connections. A CNN slides learned filters across an image, reusing weights to detect local patterns. A checkpoint stores parameters. Choose checkpoints on validation data. Dropout randomly removes activations during training; train/eval mode controls it and changes normalization behavior.

### Worked example: follow the values

For 8-by-8 grayscale images, a batch of 32 has shape `(32,1,8,8)`: batch, channels, height, width. A classifier with ten classes produces logits `(32,10)`. Each of the 32 targets is one integer from 0 through 9. Validation uses eval mode and no gradient tracking.

### Pause and explain

What happens if you call backward repeatedly without clearing gradients?

<details><summary>Check your reasoning after trying</summary>

PyTorch accumulates gradients. Unless deliberate accumulation was designed, the update no longer corresponds to just the current batch.

</details>

### Common mistakes to check

An attractive loss curve does not replace held-out metrics. Restoring the best checkpoint is a separate action from merely recording the best score.

**Ready to continue?** Explain the example without reading it, predict what a changed input will do, and identify one invalid or missing input. Then attempt the original notebook exercises. You do not need to finish a whole notebook in one session.


## Lesson

PyTorch tensors can track operations for automatic differentiation. `loss.backward()` accumulates gradients; clear them before the next update unless accumulation is intentional. `model.train()` enables training behavior such as dropout; `model.eval()` switches to evaluation behavior. `torch.no_grad()` avoids building gradient graphs during validation. These are separate controls.

An epoch visits the training set; a batch is a subset used for one update. DataLoader handles batches and shuffling. SGD, momentum, Adam, and AdamW use gradients differently; AdamW decouples weight decay. Learning rate and regularization affect fit. Normalize inputs using training statistics. Monitor train and validation loss to detect overfitting.

A CNN slides learned kernels across spatial dimensions, reusing weights. Pooling reduces spatial resolution. Images conventionally use (batch,channels,height,width). RNNs process sequences through recurring state; LSTM/GRU gates address long-range gradient problems. They provide historical context for attention.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
import torch
from torch import nn
torch.manual_seed(42)
torch.set_num_threads(1)
x = torch.tensor(3., requires_grad=True)
(x*x).backward()
print(x.grad)


## Exercise 1: Gradient with autograd

Return derivative of sum(x²) as a tensor; create a fresh float tensor with gradients enabled.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def squared_gradient(values):
    raise NotImplementedError


In [ ]:
assert torch.allclose(squared_gradient([1, -2]), torch.tensor([2., -4.]))
print("Exercise 1: checks passed")


<details><summary>Hint — open after trying</summary>

backward needs a scalar loss.

</details>


## Exercise 2: CNN shape

Return a network mapping (N,1,8,8) to (N,10): Conv2d(1,4,3,padding=1), ReLU, Flatten, Linear(256,10).

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def make_cnn():
    raise NotImplementedError


In [ ]:
assert make_cnn()(torch.zeros(2, 1, 8, 8)).shape == (2, 10)
print("Exercise 2: checks passed")


<details><summary>Hint — open after trying</summary>

Padding preserves the 8×8 spatial dimensions.

</details>


## Exercise 3: One update

Set training mode, zero gradients, compute cross entropy, backpropagate, step optimizer; return float loss.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def train_step(model, optimizer, x, y):
    raise NotImplementedError


In [ ]:
model = make_cnn()
optimizer = torch.optim.AdamW(model.parameters(), lr=.01)
x, y = torch.randn(4, 1, 8, 8), torch.tensor([0, 1, 2, 3])
before = model[-1].weight.detach().clone()
assert train_step(model, optimizer, x, y) > 0
assert not torch.equal(before, model[-1].weight)
print("Exercise 3: checks passed")


<details><summary>Hint — open after trying</summary>

Call optimizer.zero_grad before backward.

</details>


## Independent transfer

Run the digits CNN project with train/validation/test splits, DataLoader, loss curves, best-checkpoint restore, per-class errors, and a model card. Compare an MLP and CNN using the same split.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Test your independent assignment

The assignment is work you build separately from the small notebook exercises. Use your own Python file under `work/`; notebooks remain useful for exploration and notes.

1. Read the [05b interface and acceptance cases](../assignments/05b.md).
2. Copy the [blank starter](../assignments/starters/05b.py) to `work/assignments/05b.py`, or adapt your existing implementation to the documented interface. Do not overwrite earlier work.
3. Implement one piece, then run this command **in your terminal from the repository root**, with your course environment activated:

```bash
python tools/test_assignment.py 05b --solution work/assignments/05b.py
```

The runner tests the file you specify. It never silently substitutes the reference answer. `NotImplementedError` is expected while the starter is incomplete. Read a failed assertion, calculate its expected answer by hand, fix your implementation, and rerun. See [how to read tests](../assignments/README.md) if pytest is new to you.

After a serious attempt, compare the [worked implementation](../answers/05b.md#05b-t). To check the provided answer explicitly:

```bash
python tools/test_assignment.py 05b --reference
```

**Completion:** pass the relevant automated cases, complete the additional experiment/live-evidence checklist in the assignment guide, and explain your approach without the answer open. An automated pass does not certify a live server, a deployment, a human review, or independent mastery.


## Explain without code

1. Why both eval and no_grad?

2. Why can gradients explode?


## Reading and review

- [Primary reference 1](https://docs.pytorch.org/tutorials/beginner/basics/intro.html)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
